
VIDE INFORMAÇÕES DE SELEÇÃO DAS VARIAVEIS NO READEME.md

```

In [0]:
%python
# %pip install xgboost
# %pip install xgboost shap
%pip install -r ../../requirements.txt  -qqq

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%python
#  %restart_python or 
dbutils.library.restartPython() 

In [0]:
%python
# ==============================================================================
# Pipeline Databricks: Validação das Hipóteses H1, H2, H3 e H4
# via XGBoost, SHAP e simulação de threshold de decisão
#
# Cada bloco de código traz um comentário indicando exatamente qual trecho
# da hipótese (em texto) ele está respondendo, e qual parte da hipótese
# NÃO é coberta pelo teste (para não superestimar a conclusão).
# ==============================================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
import shap

# ==============================================================================
# 1. Leitura da tabela Silver (schema real, via DESCRIBE TABLE EXTENDED)
# ==============================================================================
source_table = "credito_prd.silver.give_me_some_credit"
print(f"Lendo dados de: {source_table}")

spark_df = spark.table(source_table)
df = spark_df.toPandas()

target_col = "target_dlq_2yrs"

# Colunas que NÃO são features de negócio: chave e metadado técnico do Delta/Autoloader
cols_to_ignore = [
    target_col,
    "customer_id",
    "_ingestion_timestamp",
    "_source_file",
    "_silver_processed_at",
    "_metadata",
    "_object_metadata",
]
feature_cols = [col for col in df.columns if col not in cols_to_ignore]
print(f"Features utilizadas ({len(feature_cols)}): {feature_cols}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Justificativa de Seleção de Features
# MAGIC
# MAGIC As features não foram usadas apenas porque "estavam na tabela Silver" — cada uma cobre
# MAGIC uma dimensão de risco de crédito reconhecida no domínio, e essa cobertura foi verificada
# MAGIC antes do treino. Critério de exclusão aplicado primeiro (`cols_to_ignore`): chave primária
# MAGIC (`customer_id`), metadados técnicos do pipeline Delta/Autoloader e o target — nenhum é
# MAGIC feature de negócio, e mantê-los geraria vazamento de dado ou ruído sem valor preditivo.
# MAGIC
# MAGIC Das 10 features restantes, o agrupamento por dimensão de risco é:
# MAGIC
# MAGIC | Dimensão de risco | Features | Racional |
# MAGIC |---|---|---|
# MAGIC | Uso do crédito disponível | `revolving_utilization` | Quanto mais perto do limite, menor a folga financeira do cliente (base da H1). |
# MAGIC | Histórico de atraso | `num_times_30_59_days_late`, `num_times_60_89_days_late`, `num_times_90_days_late` | Comportamento passado de pagamento é historicamente o preditor mais forte em risco de crédito; mantidas separadas por faixa de severidade em vez de agregadas, para não perder sinal de gravidade. |
# MAGIC | Capacidade financeira | `debt_ratio`, `monthly_income` | Dívida relativa à renda só é interpretável junto da renda absoluta — dois clientes com o mesmo `debt_ratio` podem ter capacidade real muito diferente (base da H2, comparada a `age`). |
# MAGIC | Exposição e perfil de vida | `num_open_credit_lines`, `num_real_estate_loans`, `num_dependents`, `age` | Quantidade de compromissos simultâneos, presença de financiamento imobiliário (colateral, geralmente reduz risco) e estrutura familiar/estágio de vida (impacta despesa fixa e maturidade de crédito). |
# MAGIC
# MAGIC **Validações aplicadas antes de fechar a lista:**
# MAGIC - Nenhuma feature é constante ou tem nulo excessivo a ponto de inviabilizar o campo
# MAGIC   (exceção conhecida: `monthly_income` e `num_dependents`, tratadas via imputação por
# MAGIC   mediana no pré-processamento, ver Seção 3).
# MAGIC - Cada feature está amarrada a pelo menos uma hipótese testável (H1–H4) documentada no
# MAGIC   código abaixo — evita incluir variável "porque sim", sem propósito analítico claro.
# MAGIC - **Não testado nesta versão:** colinearidade entre as três variáveis de atraso
# MAGIC   (`num_times_*_days_late`), que pode inflar ou distorcer a importância relativa no SHAP.
# MAGIC   Fica como limitação registrada, não como validação feita.



# X = df[feature_cols]
# y = df[target_col].astype(int)

# Diagnóstico: quantos registros têm target nulo/infinito?
n_total = len(df)
target_invalido = df[target_col].isna() | np.isinf(df[target_col])
print(f"Registros com {target_col} nulo/inválido: {int(target_invalido.sum())} de {n_total}")

# Descarta linhas sem rótulo — não é possível treinar supervisionado sem target
df_valid = df[~target_invalido].copy()
X = df_valid[feature_cols]
y = df_valid[target_col].astype(int)


# ==============================================================================
# 2. Split Estratificado (Treino / Teste) antes de qualquer transformação
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ==============================================================================
# 3. Pré-processamento sem vazamento de dado (fit só no treino)
# ==============================================================================
cols_with_nulls = [c for c in ["monthly_income", "num_dependents"] if c in feature_cols]
other_numeric_cols = [c for c in feature_cols if c not in cols_with_nulls]

preprocessor = ColumnTransformer(
    transformers=[
        ("impute_median", SimpleImputer(strategy="median"), cols_with_nulls),
        ("passthrough_numeric", SimpleImputer(strategy="median"), other_numeric_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

transformed_feature_names = list(preprocessor.get_feature_names_out())
X_train_df = pd.DataFrame(X_train_prep, columns=transformed_feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_prep, columns=transformed_feature_names, index=X_test.index)

# ==============================================================================
# 4. Treinamento: XGBoost (principal) + Regressão Logística (baseline)
# ==============================================================================
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight_value = float(neg_count / pos_count)
print(f"Total Negativos: {neg_count} | Total Positivos: {pos_count}")
print(f"scale_pos_weight configurado: {scale_pos_weight_value:.2f}")

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    scale_pos_weight=scale_pos_weight_value,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="auc",
)
xgb_model.fit(X_train_df, y_train)

logreg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logreg_pipeline.fit(X_train_df, y_train)

y_pred_xgb = xgb_model.predict_proba(X_test_df)[:, 1]
y_pred_logreg = logreg_pipeline.predict_proba(X_test_df)[:, 1]

print(f"\nROC-AUC XGBoost: {roc_auc_score(y_test, y_pred_xgb):.4f}")
print(f"ROC-AUC Regressão Logística: {roc_auc_score(y_test, y_pred_logreg):.4f}")

# ==============================================================================
# 5. SHAP — base para H1 e H2
# ==============================================================================
explainer = shap.TreeExplainer(xgb_model)
shap_sample = X_test_df.sample(n=min(2000, len(X_test_df)), random_state=42)
shap_values = explainer(shap_sample)

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    "feature": transformed_feature_names,
    "mean_abs_shap": mean_abs_shap,
}).sort_values(by="mean_abs_shap", ascending=False).reset_index(drop=True)

logreg_coefs = pd.DataFrame({
    "feature": transformed_feature_names,
    "coeficiente_logreg": logreg_pipeline.named_steps["logreg"].coef_[0],
}).sort_values(by="coeficiente_logreg", ascending=False).reset_index(drop=True)

print("\n--- Ranking de Importância SHAP (Top Features) ---")
print(shap_importance_df)

# ==============================================================================================
# H1 — Utilização de crédito rotativo:
# "clientes com maior RevolvingUtilizationOfUnsecuredLines (percentual do limite de crédito já
#  utilizado) têm probabilidade significativamente maior de inadimplência nos próximos 2 anos,
#  porque um uso próximo do limite indica menor folga financeira."
# ==============================================================================================
FEATURE_H1 = "revolving_utilization"

# "...têm probabilidade [...] maior de inadimplência..."
# -> Correlação entre o VALOR da feature e o SHAP daquela observação.
#    Se correlação > 0: quando revolving_utilization sobe, o SHAP também sobe,
#    ou seja, o modelo empurra a previsão PARA inadimplência quando a utilização é alta.
#    Isso responde a DIREÇÃO do efeito descrito na hipótese.
corr_util = float(
    np.corrcoef(shap_sample[FEATURE_H1], shap_values[:, FEATURE_H1].values)[0, 1]
)

pos_util = int(shap_importance_df[shap_importance_df["feature"] == FEATURE_H1].index[0] + 1)
util_shap_val = float(
    shap_importance_df.loc[shap_importance_df["feature"] == FEATURE_H1, "mean_abs_shap"].values[0]
)

# "...probabilidade SIGNIFICATIVAMENTE maior..."
# -> A palavra "significativamente" pede mais do que sinal positivo: pede uma correlação
#    forte o suficiente para não ser ruído. Por isso usamos um limiar mínimo (CORR_THRESHOLD),
#    em vez de aceitar qualquer corr_util > 0. Isso é o que torna o teste rigoroso, não cosmético.
CORR_THRESHOLD = 0.05
h1_status = "Confirmada" if corr_util > CORR_THRESHOLD else "Contrariada / Inconclusiva"

# NÃO RESPONDIDO por este teste:
# -> "...porque um uso próximo do limite indica menor folga financeira."
#    Essa é a explicação CAUSAL da hipótese. O SHAP mostra ASSOCIAÇÃO (o modelo usa essa
#    variável e na direção esperada), mas não prova que "menor folga financeira" é o mecanismo
#    real por trás disso — isso exigiria um estudo causal (ex: variável instrumental), fora
#    do escopo deste script. No relatório para o CEO, isso deve ficar explícito como limitação.
print("\n================== VALIDAÇÃO H1 ==================")
print(f"Variável: {FEATURE_H1}")
print(f"Posição no ranking SHAP: {pos_util}º lugar (Impacto médio: {util_shap_val:.4f})")
print(f"Correlação Feature vs SHAP: {corr_util:.4f} (limiar de significância prática: {CORR_THRESHOLD})")
print(f"Conclusão H1 (direção e força do efeito): {h1_status}")
print("Não testado: o mecanismo causal ('porque indica menor folga financeira').")

# ==============================================================================================
# H2 — Dívida/renda como preditor dominante:
# "o DebtRatio (relação dívida/renda) tem poder preditivo maior que a idade do cliente
#  isoladamente — ou seja, a capacidade de pagamento pesa mais na inadimplência do que
#  o perfil demográfico."
# ==============================================================================================
FEATURE_H2_A = "debt_ratio"
FEATURE_H2_B = "age"

debtratio_shap = float(
    shap_importance_df.loc[shap_importance_df["feature"] == FEATURE_H2_A, "mean_abs_shap"].values[0]
)
age_shap = float(
    shap_importance_df.loc[shap_importance_df["feature"] == FEATURE_H2_B, "mean_abs_shap"].values[0]
)
debtratio_pos = int(shap_importance_df[shap_importance_df["feature"] == FEATURE_H2_A].index[0] + 1)
age_pos = int(shap_importance_df[shap_importance_df["feature"] == FEATURE_H2_B].index[0] + 1)

# "...DebtRatio [...] tem poder preditivo MAIOR QUE a idade [...] isoladamente..."
# -> Comparação direta do |SHAP| médio das duas variáveis: quem contribui mais, em média,
#    para as previsões do modelo. Essa é a leitura literal de "poder preditivo maior".
#    Usamos diferença RELATIVA (não só ">") para exigir uma margem mínima, evitando
#    declarar "confirmada" com uma diferença desprezível (ex: 0.001 vs 0.0009).
RELATIVE_MARGIN = 0.10  # debt_ratio precisa superar age em pelo menos 10%
relative_diff = (debtratio_shap - age_shap) / age_shap if age_shap > 0 else float("inf")
h2_status = "Confirmada" if relative_diff > RELATIVE_MARGIN else "Contrariada / Inconclusiva"

# PARCIALMENTE RESPONDIDO:
# -> "...a capacidade de pagamento pesa mais [...] do que o PERFIL DEMOGRÁFICO."
#    A hipótese generaliza para "perfil demográfico", mas o teste só compara com `age`.
#    Se o dataset tiver outra variável demográfica relevante (ex: num_dependents), o teste
#    atual NÃO cobre essa comparação — está limitado ao que está explícito entre parênteses
#    na própria hipótese ("...que a idade do cliente isoladamente").
print("\n================== VALIDAÇÃO H2 ==================")
debtratio_coef = float(logreg_coefs.loc[logreg_coefs["feature"] == FEATURE_H2_A, "coeficiente_logreg"].values[0])
age_coef = float(logreg_coefs.loc[logreg_coefs["feature"] == FEATURE_H2_B, "coeficiente_logreg"].values[0])
print(f"debt_ratio -> Ranking SHAP: {debtratio_pos}º (|SHAP|: {debtratio_shap:.4f}) | Coef RegLog: {debtratio_coef:.4f}")
print(f"age        -> Ranking SHAP: {age_pos}º (|SHAP|: {age_shap:.4f}) | Coef RegLog: {age_coef:.4f}")
print(f"Diferença relativa (debt_ratio vs age): {relative_diff:.2%} (limiar exigido: {RELATIVE_MARGIN:.0%})")
print(f"Conclusão H2 (debt_ratio vs age): {h2_status}")
print("Cobertura parcial: compara só com 'age', não com todo o 'perfil demográfico'.")

# ==============================================================================================
# H3 — Threshold ótimo de aprovação:
# "Existe um ponto de corte de probabilidade que reduz a inadimplência da carteira aprovada
#  sem derrubar a taxa de aprovação total abaixo de um piso viável para o negócio."
# ==============================================================================================

# Piso mínimo de aprovação que o negócio consideraria aceitável (ajustável conforme política
# de crédito real da Mezzo — aqui usamos 70% como piso ilustrativo).
MIN_APPROVAL_RATE = 0.70

# Taxa de inadimplência da "política atual" = aprovar todo mundo (baseline sem modelo),
# ou seja, a taxa de inadimplência real da base de teste sem nenhum filtro.
baseline_default_rate = float(y_test.mean())

# "...existe um ponto de corte..."
# -> Varremos vários thresholds de probabilidade prevista (y_pred_xgb) e, para cada um,
#    calculamos: (a) taxa de aprovação resultante (quantos clientes ficam abaixo do corte,
#    ou seja, são aprovados) e (b) taxa de inadimplência DENTRO dos aprovados.
thresholds = np.arange(0.05, 0.95, 0.05)
threshold_results = []
for t in thresholds:
    approved_mask = y_pred_xgb < t  # abaixo do corte de risco = aprovado
    approval_rate = float(approved_mask.mean())
    if approved_mask.sum() > 0:
        default_rate_approved = float(y_test[approved_mask].mean())
    else:
        default_rate_approved = np.nan
    threshold_results.append({
        "threshold": round(float(t), 2),
        "approval_rate": approval_rate,
        "default_rate_approved": default_rate_approved,
    })

threshold_df = pd.DataFrame(threshold_results)

# "...reduz a inadimplência da carteira aprovada..."
# -> Filtramos apenas thresholds que respeitam o piso de aprovação do negócio, e entre esses,
#    escolhemos o que traz a MENOR taxa de inadimplência na carteira aprovada.
viable = threshold_df[threshold_df["approval_rate"] >= MIN_APPROVAL_RATE]

# "...sem derrubar a taxa de aprovação [...] abaixo de um piso viável..."
# -> Esse filtro (approval_rate >= MIN_APPROVAL_RATE) é exatamente a restrição de negócio
#    da hipótese: não adianta reduzir inadimplência a zero rejeitando todo mundo.
if len(viable) > 0:
    best_row = viable.sort_values("default_rate_approved").iloc[0]
    reduction = baseline_default_rate - best_row["default_rate_approved"]
    h3_status = "Confirmada" if reduction > 0 else "Contrariada / Inconclusiva"
else:
    best_row = None
    reduction = None
    h3_status = "Inconclusiva (nenhum threshold atinge o piso de aprovação definido)"

print("\n================== VALIDAÇÃO H3 ==================")
print(f"Taxa de inadimplência da base (sem filtro / política atual simulada): {baseline_default_rate:.2%}")
print(f"Piso de aprovação exigido pelo negócio: {MIN_APPROVAL_RATE:.0%}")
if best_row is not None:
    print(f"Melhor threshold encontrado: {best_row['threshold']}")
    print(f"  -> Taxa de aprovação resultante: {best_row['approval_rate']:.2%}")
    print(f"  -> Taxa de inadimplência na carteira aprovada: {best_row['default_rate_approved']:.2%}")
    print(f"  -> Redução de inadimplência vs. baseline: {reduction:.2%}")
print(f"Conclusão H3: {h3_status}")
print("Não testado: estabilidade desse threshold ao longo do tempo (isso seria H6, drift).")

# ==============================================================================================
# H4 — Impacto financeiro vs. política atual:
# "O modelo, aplicado retroativamente à base histórica, teria evitado um volume de perdas em
#  R$ maior do que o custo de rejeitar bons pagadores (falsos positivos)."
# ==============================================================================================

# Valores de referência ILUSTRATIVOS — em um caso real, estes viriam de dados financeiros
# reais da Mezzo (perda média por inadimplência, margem média por cliente aprovado, etc.)
PERDA_MEDIA_POR_INADIMPLENCIA_R = 8000.0   # custo médio de um cliente que dá default
MARGEM_MEDIA_POR_CLIENTE_BOM_R = 1200.0    # receita/margem perdida ao rejeitar um bom pagador

# Usamos o mesmo threshold escolhido em H3 como "a política proposta pelo modelo"
if best_row is not None:
    t_final = best_row["threshold"]
    approved_mask_final = y_pred_xgb < t_final

    # "...evitado um volume de perdas em R$..."
    # -> Matriz de confusão: quantos maus pagadores (y_test == 1) o modelo teria REJEITADO
    #    (approved_mask_final == False) que a política atual (aprovar todo mundo) teria aceitado.
    maus_pagadores_evitados = int(((y_test == 1) & (~approved_mask_final)).sum())
    perda_evitada_R = maus_pagadores_evitados * PERDA_MEDIA_POR_INADIMPLENCIA_R

    # "...custo de rejeitar bons pagadores (falsos positivos)."
    # -> Bons pagadores (y_test == 0) que o modelo também rejeitou junto (falso positivo
    #    do ponto de vista de "vou negar crédito a alguém que pagaria certinho").
    bons_pagadores_rejeitados = int(((y_test == 0) & (~approved_mask_final)).sum())
    custo_oportunidade_R = bons_pagadores_rejeitados * MARGEM_MEDIA_POR_CLIENTE_BOM_R

    # "...maior do que o custo..."
    # -> Comparação direta: perda evitada (ganho) vs. custo de oportunidade (perda).
    impacto_liquido_R = perda_evitada_R - custo_oportunidade_R
    h4_status = "Confirmada" if impacto_liquido_R > 0 else "Contrariada / Inconclusiva"

    print("\n================== VALIDAÇÃO H4 ==================")
    print(f"Threshold utilizado (herdado de H3): {t_final}")
    print(f"Maus pagadores evitados: {maus_pagadores_evitados} -> Perda evitada: R$ {perda_evitada_R:,.2f}")
    print(f"Bons pagadores rejeitados (falso positivo): {bons_pagadores_rejeitados} -> Custo de oportunidade: R$ {custo_oportunidade_R:,.2f}")
    print(f"Impacto financeiro líquido: R$ {impacto_liquido_R:,.2f}")
    print(f"Conclusão H4: {h4_status}")
    print("ATENÇÃO: PERDA_MEDIA_POR_INADIMPLENCIA_R e MARGEM_MEDIA_POR_CLIENTE_BOM_R são valores")
    print("ilustrativos. Para uso real, substituir pelos valores financeiros reais da Mezzo.")
else:
    h4_status = "Não calculada (H3 não encontrou threshold viável)"
    perda_evitada_R = None
    custo_oportunidade_R = None
    impacto_liquido_R = None
    print("\n================== VALIDAÇÃO H4 ==================")
    print("H4 depende de H3. Como H3 foi inconclusiva, H4 também não pôde ser calculada.")

# ==============================================================================
# Persistência de TODAS as hipóteses na camada Gold
# ==============================================================================
gold_hypotheses_df = pd.DataFrame([
    {
        "hipotese": "H1",
        "descricao": "Clientes com maior revolving_utilization têm maior probabilidade de inadimplência",
        "metrica": corr_util,
        "status": h1_status,
        "evidencia": f"Correlação SHAP-Feature de {corr_util:.4f} (limiar: {CORR_THRESHOLD}). Posição {pos_util}º no ranking SHAP.",
        "limitacao": "Não testa o mecanismo causal ('menor folga financeira'), apenas associação via SHAP.",
    },
    {
        "hipotese": "H2",
        "descricao": "debt_ratio tem poder preditivo maior que age isoladamente",
        "metrica": relative_diff,
        "status": h2_status,
        "evidencia": f"debt_ratio SHAP ({debtratio_shap:.4f}) vs age SHAP ({age_shap:.4f}). Diferença relativa: {relative_diff:.2%} (limiar: {RELATIVE_MARGIN:.0%}).",
        "limitacao": "Compara só com 'age', não com o 'perfil demográfico' como um todo.",
    },
    {
        "hipotese": "H3",
        "descricao": "Existe threshold que reduz inadimplência da carteira aprovada sem violar piso de aprovação",
        "metrica": reduction if reduction is not None else np.nan,
        "status": h3_status,
        "evidencia": (
            f"Threshold {best_row['threshold']}, aprovação {best_row['approval_rate']:.2%}, "
            f"inadimplência aprovados {best_row['default_rate_approved']:.2%}"
            if best_row is not None else "Nenhum threshold atingiu o piso de aprovação definido."
        ),
        "limitacao": "Não testa estabilidade do threshold ao longo do tempo (drift).",
    },
    {
        "hipotese": "H4",
        "descricao": "Impacto financeiro líquido de aplicar o modelo (perda evitada vs. custo de oportunidade)",
        "metrica": impacto_liquido_R if impacto_liquido_R is not None else np.nan,
        "status": h4_status,
        "evidencia": (
            f"Perda evitada: R$ {perda_evitada_R:,.2f} | Custo de oportunidade: R$ {custo_oportunidade_R:,.2f}"
            if impacto_liquido_R is not None else "Dependente de H3."
        ),
        "limitacao": "Valores de perda/margem são ilustrativos — substituir por dados financeiros reais da Mezzo.",
    },
])

target_gold_table = "credito_prd.gold.gold_hypotheses_validation"
spark_gold_df = spark.createDataFrame(gold_hypotheses_df)
spark_gold_df.write.mode("overwrite").format("delta").saveAsTable(target_gold_table)

print(f"\nTabela Delta salva com sucesso no catálogo: {target_gold_table}")

Lendo dados de: credito_prd.silver.give_me_some_credit
Features utilizadas (10): ['revolving_utilization', 'age', 'num_times_30_59_days_late', 'debt_ratio', 'monthly_income', 'num_open_credit_lines', 'num_times_90_days_late', 'num_real_estate_loans', 'num_times_60_89_days_late', 'num_dependents']
Registros com target_dlq_2yrs nulo/inválido: 1 de 150000
Total Negativos: 111978 | Total Positivos: 8021
scale_pos_weight configurado: 13.96

ROC-AUC XGBoost: 0.8617
ROC-AUC Regressão Logística: 0.7811

--- Ranking de Importância SHAP (Top Features) ---
                     feature  mean_abs_shap
0      revolving_utilization       0.834482
1  num_times_30_59_days_late       0.411382
2     num_times_90_days_late       0.311290
3                        age       0.246558
4      num_open_credit_lines       0.203764
5  num_times_60_89_days_late       0.180782
6                 debt_ratio       0.141995
7             monthly_income       0.121973
8      num_real_estate_loans       0.100859
9       